In [38]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
from datasets import Dataset, DatasetDict

# Load dataset
data = pd.read_csv('/kaggle/input/bhaav-dataset/bhaav.csv')
train_df, validation_df = train_test_split(data, test_size=0.3, stratify=data['label'])
dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df),
    'validation': Dataset.from_pandas(validation_df)
})

# Define label mapping
num_labels = 4

# Load tokenizer and model
# model_name = '/kaggle/input/xlm-roberta-base/xlm-roberta-base'
model_name = '/kaggle/working/bhavnai-model'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Tokenize the text
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    # Checkpoints
    save_strategy='epoch',
    output_dir='/kaggle/working/bhavnai-output',
    save_total_limit=1,  # Keeps only the latest checkpoint(s)

    # Logging
    logging_strategy='no',
    # logging_dir='/kaggle/working/logs',

    # Learning hyperparameters
    eval_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    report_to='none',
    fp16=True,
    warmup_steps=500,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    resume_from_checkpoint=True,
)

# DataCollator for preparing batches of data during training or evaluation
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='macro')
    return {
        'accuracy': acc,
        'f1': f1
    }

# Weighted trainer for unbalanced datasets
class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            # Ensure class_weights is on the correct device when initializing
            # It's better to move it inside compute_loss if inputs.device might change,
            # but moving it here is often fine if you know the device beforehand.
            # Let's move it to compute_loss for robustness.
            # self.class_weights = class_weights.to(self.args.device) # Alternative placement
            self.class_weights_cpu = class_weights # Keep on CPU initially
        else:
            self.class_weights_cpu = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')

        # Ensure class weights are on the same device as the inputs/model
        effective_class_weights = None
        if self.class_weights_cpu is not None:
             effective_class_weights = self.class_weights_cpu.to(inputs['input_ids'].device)

        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits

        # Compute custom loss
        loss_fct = torch.nn.CrossEntropyLoss(weight=effective_class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# Calculate class weights - move to correct device inside Trainer
class_weights = compute_class_weight('balanced', classes=np.unique(dataset['train']['label']), y=dataset['train']['label'])
# Keep weights on CPU initially, let Trainer handle device placement
class_weights = torch.tensor(class_weights).float()

# Define Trainer
trainer = WeightedTrainer(
    class_weights=class_weights, # Use weighted class label update mechanism
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

# Train the model
trainer.train()

# Save model & tokenizer
trainer.save_model('/kaggle/working/bhavnai-model')
tokenizer.save_pretrained('/kaggle/working/bhavnai-model')

Map:   0%|          | 0/13157 [00:00<?, ? examples/s]

Map:   0%|          | 0/5639 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.773541,0.830644,0.797402
2,No log,0.594991,0.761660,0.733393
3,No log,0.664069,0.816279,0.772467
4,No log,0.774973,0.796595,0.762396
5,No log,0.780186,0.796772,0.749967
6,No log,0.851260,0.826033,0.777747


('/kaggle/working/bhavnai-model/tokenizer_config.json',
 '/kaggle/working/bhavnai-model/special_tokens_map.json',
 '/kaggle/working/bhavnai-model/sentencepiece.bpe.model',
 '/kaggle/working/bhavnai-model/added_tokens.json',
 '/kaggle/working/bhavnai-model/tokenizer.json')

In [39]:
from sklearn.metrics import classification_report
preds_output = trainer.predict(tokenized_dataset['validation'])
print(classification_report(preds_output.label_ids, np.argmax(preds_output.predictions, axis=1)))
pd.DataFrame([trainer.evaluate()]).T

              precision    recall  f1-score   support

           0       0.82      0.78      0.80       440
           1       0.69      0.85      0.76       951
           2       0.92      0.83      0.88      3509
           3       0.68      0.83      0.75       739

    accuracy                           0.83      5639
   macro avg       0.78      0.82      0.80      5639
weighted avg       0.85      0.83      0.83      5639



,0
eval_loss,0.773541
eval_accuracy,0.830644
eval_f1,0.797402
eval_runtime,25.338800
eval_samples_per_second,222.544000
eval_steps_per_second,27.823000
epoch,6.000000


In [90]:
import re
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# Load your fine-tuned model and tokenizer
model_path = './bhavnai-model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Define your label mapping (index to label)
labels = ['Angry 😠', 'Sad 😢', 'Neutral 😐', 'Happy 😊']

# Input text
input_text = 'graahaka sahaayataa kitnii bekaara hai yaahaa!'

processed_text = transliterate(input_text, sanscript.ITRANS, sanscript.DEVANAGARI)

# Tokenize input
inputs = tokenizer(processed_text, return_tensors='pt', truncation=True, padding=True)

# Predict
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)
    probs = F.softmax(outputs.logits, dim=-1)
    attentions = outputs.attentions

    # Calculate the average attention across all layers and heads
    attentions_avg = torch.mean(attentions[-1], dim=(1, 2))  # Using the last layer for simplicity

    # Map the attention scores to token positions
    attention_scores = attentions_avg.squeeze().detach().cpu().numpy()

    # Get token ids and convert them back to text
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze().cpu().numpy())

    # Extract keywords from the processed text based on attention values of tokens
    keywords = {kw: 0 for kw in re.findall('[।.,?!:;"\'\(\)\-]|[^।.,?!:;"\'\(\)\-\s]+', processed_text) if len(kw) != 0}
    for t, a in zip(tokens, attention_scores):
        for kw in keywords.keys():
            if t.lstrip('▁') in kw:
                keywords[kw] += a
                break
    if len(keywords) != 0:
        keywords = [kw for kw, _ in sorted(keywords.items(), key=lambda item: item[1], reverse=True)]

predicted_output = {
    'text': processed_text,
    'sentiment': labels[probs.argmax().item()],
    'confidence': probs.max().item(),
    'keywords': keywords[:5] if keywords else [],
}

print(f'Predicted output: {predicted_output}')

Predicted output: {'text': 'ग्राहक सहायता कित्नी बेकार है याहा!', 'sentiment': 'Sad 😢', 'confidence': 0.9926713705062866, 'keywords': ['कित्नी', 'सहायता', 'याहा', 'बेकार', '!']}


In [91]:
!zip -r output_files.zip /kaggle/working/
from IPython.display import FileLink
FileLink(r'output_files.zip')

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/bhavnai-model/ (stored 0%)
  adding: kaggle/working/bhavnai-model/sentencepiece.bpe.model

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 49%)
  adding: kaggle/working/bhavnai-model/config.json (deflated 53%)
  adding: kaggle/working/bhavnai-model/model.safetensors (deflated 29%)
  adding: kaggle/working/bhavnai-model/training_args.bin (deflated 51%)
  adding: kaggle/working/bhavnai-model/tokenizer_config.json (deflated 74%)
  adding: kaggle/working/bhavnai-model/special_tokens_map.json (deflated 85%)
  adding: kaggle/working/bhavnai-model/tokenizer.json (deflated 76%)
  adding: kaggle/working/bhavnai-output/ (stored 0%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/ (stored 0%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/sentencepiece.bpe.model (deflated 49%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/optimizer.pt (deflated 70%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/trainer_state.json (deflated 59%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/config.json (deflated 53%)
  adding: kaggle/working/bhavnai-output/checkpoint-412/model.safetensors (deflated

/kaggle/working/output_files.zip